# Class 7: Risk

**GIS Vulnerability & Risk Assessment Course**

## Class Objectives
By the end of this lesson, you will be able to:
- Understand what **risk** means in a vulnerability assessment
- Read and apply a **risk scoring matrix** to combine probability and consequence
- Calculate risk scores for geographic features using a lookup matrix
- Visualize risk levels on a map
- Generate risk statistics and save results to a GeoPackage

## What is Risk?

**Risk** is the combination of two factors:
- **Probability**: How likely is this hazard to happen? (How often? How certain?)
- **Consequence**: How bad would it be if it happened? (What would be the impact?)

**Risk = Probability × Consequence**

A rare but catastrophic event can be a high risk. A frequent but minor event can also be a high risk. This lesson combines these two measures using a structured decision table called a **Risk Scoring Matrix**.


## Step 1: Install and Import Libraries

We will use Python libraries to work with geographic data:
- **GeoPandas**: For reading and writing geographic data (shapefiles, GeoPackages)
- **Pandas**: For working with tables of data
- **Matplotlib**: For making charts and maps
- **Folium**: For interactive maps
- **Fiona**: For reading geographic file formats

Run the cell below to install these libraries (this may take a minute).


In [ ]:
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet
print("✓ Libraries installed successfully!")

Now import the libraries we just installed:


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")


## Step 2: Connect to Google Drive

Google Colab runs in the cloud. To access your files, we need to "mount" (connect to) your Google Drive. Run the cell below and follow the instructions to authorize Colab.


In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading
INPUT_GPKG = os.path.join(DATA_DIR, 'class_6_consequence.gpkg')  # Load from Class 6
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_7_risk.gpkg')  # Save to Class 7
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

## Step 4: Understanding the Risk Scoring Matrix

The **Risk Scoring Matrix** is a lookup table that combines probability and consequence into a risk score. Here is how it works:

| | **Probability: Low (1)** | **Probability: Medium (2)** | **Probability: High (3)** |
|---|---|---|---|
| **Consequence: High (3)** | 2 (Medium) | 3 (High) | 3 (High) |
| **Consequence: Medium (2)** | 1 (Low) | 2 (Medium) | 3 (High) |
| **Consequence: Low (1)** | 1 (Low) | 1 (Low) | 2 (Medium) |

### How to Read the Matrix:
1. Find the row that matches your **consequence** level (top of table)
2. Find the column that matches your **probability** level (left of table)
3. The number at the intersection is your **risk score**

### Examples:
- **High consequence + High probability = High risk (3)** — This is very concerning! The hazard is likely to happen AND would cause major damage.
- **High consequence + Low probability = Medium risk (2)** — The impact would be severe, but it's unlikely to happen.
- **Low consequence + Low probability = Low risk (1)** — Minor potential impact AND it's unlikely. Not a concern.
- **Medium consequence + Medium probability = Medium risk (2)** — Moderate hazard level.

**Important Note:** This matrix is **different from the Vulnerability Matrix** in Class 4. The rows are ordered **High to Low** (from top to bottom), whereas the vulnerability matrix rows were ordered **Low to High**. Be careful when reading the correct row!


## Step 5: Visualize the Risk Scoring Matrix

The cell below creates a color-coded version of the risk matrix. This visual makes it easy to see which combinations are low, medium, or high risk at a glance:
- **Light lavender** = Low risk (1)
- **Light purple** = Medium risk (2)
- **Medium purple** = High risk (3)

Study this diagram carefully. You will use it when you calculate risk scores for your parcels!

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# Risk matrix: rows are Consequence (High=top, Low=bottom), cols are Probability (Low, Medium, High)
# Key: (consequence, probability) → risk
risk_matrix = np.array([[3, 3, 3],   # Consequence High
                        [1, 2, 3],   # Consequence Medium
                        [1, 1, 2]])  # Consequence Low

# Color mapping: 1=light purple, 2=medium purple, 3=dark purple (official risk colors)
color_map = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}
text_colors = {1: 'black', 2: 'black', 3: 'white'}
risk_labels = {1: 'Low\n(1)', 2: 'Medium\n(2)', 3: 'High\n(3)'}

# Draw the matrix
for i in range(3):
    for j in range(3):
        risk_val = risk_matrix[i, j]
        color = color_map[risk_val]

        # Draw rectangle (cell)
        rect = plt.Rectangle((j, 2-i), 1, 1, linewidth=2, edgecolor='black', facecolor=color)
        ax.add_patch(rect)

        # Add text in the center of the cell
        text_color = text_colors[risk_val]
        ax.text(j+0.5, 2-i+0.5, risk_labels[risk_val],
                ha='center', va='center', fontsize=14, fontweight='bold', color=text_color)

# Set up axes
ax.set_xlim(0, 3)
ax.set_ylim(0, 3)
ax.set_aspect('equal')

# Column labels (Probability)
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels(['Low\n(1)', 'Medium\n(2)', 'High\n(3)'], fontsize=12, fontweight='bold')

# Row labels (Consequence - reversed because we draw top-to-bottom)
ax.set_yticks([0.5, 1.5, 2.5])
ax.set_yticklabels(['Low\n(1)', 'Medium\n(2)', 'High\n(3)'], fontsize=12, fontweight='bold')

# Axis labels
ax.set_xlabel('Probability →', fontsize=13, fontweight='bold')
ax.set_ylabel('Consequence ↓', fontsize=13, fontweight='bold')

# Title
ax.set_title('Risk Scoring Matrix\n(Combination of Probability & Consequence)',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("✓ Risk matrix visualization complete!")
print("\nUse this matrix to look up risk scores:")
print("  1. Find the row for your consequence level")
print("  2. Find the column for your probability level")
print("  3. The cell at the intersection is your risk score")

## Step 6: Example Walkthrough

Let's work through some examples to practice reading the matrix:

### Example 1: Parcel in High-Risk Area
- **Consequence**: High (3) — The building is valuable and many people live there
- **Probability**: High (3) — The parcel is in the floodway; floods happen often
- **Risk**: Find row "High" and column "High" → **Risk = High (3)** ✓

**Interpretation**: This is a very serious risk. The area floods frequently AND the impact would be severe. Action needed!

### Example 2: Parcel in Unlikely but Dangerous Zone
- **Consequence**: High (3) — Large industrial facility with hazardous materials
- **Probability**: Low (1) — Earthquakes are rare in this region
- **Risk**: Find row "High" and column "Low" → **Risk = Medium (2)** ✓

**Interpretation**: Even though earthquakes are unlikely, if one occurred, the damage would be catastrophic. This still needs monitoring.

### Example 3: Low-Risk Parcel
- **Consequence**: Low (1) — Empty lot, no structures or people
- **Probability**: Medium (2) — Hazard occurs sometimes
- **Risk**: Find row "Low" and column "Medium" → **Risk = Low (1)** ✓

**Interpretation**: Even though the hazard happens occasionally, it doesn't matter much because there is nothing valuable at risk here.

### Example 4: Medium Risk
- **Consequence**: Medium (2) — Modest residential property
- **Probability**: Medium (2) — Moderate flood frequency (50-year flood zone)
- **Risk**: Find row "Medium" and column "Medium" → **Risk = Medium (2)** ✓

**Interpretation**: Moderate risk. Needs some attention but not urgent.

These examples show how **both probability AND consequence matter**. A rare but catastrophic event (Example 2) can be higher risk than a frequent but minor event.


## Step 7: Load Parcel Data from GeoPackage

Now we load your parcel data from the GeoPackage file. This file contains polygons (parcel boundaries) and columns for probability and consequence scores that were calculated in earlier classes.

The cell below:
1. Opens the GeoPackage file
2. Reads the parcel layer
3. Displays the first few rows to show what data we have


In [ ]:
# Read the parcels layer from INPUT GeoPackage (from Class 6)
parcels = gpd.read_file(INPUT_GPKG, layer='parcels')

print(f"\u2713 Loaded {len(parcels)} parcels from Class 6")
print(f"\nColumns in dataset:")
print(parcels.columns.tolist())

# Verify the per-asset scoring columns exist:
#   consequence_a1, probability_a1, consequence_a2, probability_a2
#   is_asset_1, is_asset_2
expected_cols = ['consequence_a1', 'probability_a1', 'consequence_a2', 'probability_a2',
                 'is_asset_1', 'is_asset_2']
for col in expected_cols:
    status = '\u2713' if col in parcels.columns else '\u2717 MISSING'
    print(f"  {status}  {col}")

print(f"\nFirst 5 rows (asset scoring columns):")
show_cols = [c for c in ['FID', 'parno', 'is_asset_1', 'consequence_a1', 'probability_a1',
                          'is_asset_2', 'consequence_a2', 'probability_a2'] if c in parcels.columns]
print(parcels[show_cols].head())


## Step 8: Set Up the Risk Lookup Dictionary

Now we define the risk matrix as a Python dictionary. A dictionary stores key-value pairs, making it easy to look up the risk score for any combination of probability and consequence.

The format is:
```
(consequence, probability) → risk_score
```

For example:
- `(3, 3)` means consequence=3 AND probability=3 → risk=3 (High)
- `(2, 2)` means consequence=2 AND probability=2 → risk=2 (Medium)
- `(1, 1)` means consequence=1 AND probability=1 → risk=1 (Low)

This dictionary makes the next step very simple!


In [ ]:
# Define the risk lookup matrix
# Key: (consequence, probability) → Value: risk_score
risk_matrix_dict = {
    (3, 1): 2,  # High consequence, Low probability → Medium risk
    (3, 2): 3,  # High consequence, Medium probability → High risk
    (3, 3): 3,  # High consequence, High probability → High risk
    (2, 1): 1,  # Medium consequence, Low probability → Low risk
    (2, 2): 2,  # Medium consequence, Medium probability → Medium risk
    (2, 3): 3,  # Medium consequence, High probability → High risk
    (1, 1): 1,  # Low consequence, Low probability → Low risk
    (1, 2): 1,  # Low consequence, Medium probability → Low risk
    (1, 3): 2,  # Low consequence, High probability → Medium risk
}

print("✓ Risk matrix dictionary created")
print("\nExample lookups:")
print(f"  Consequence=3, Probability=3 → Risk = {risk_matrix_dict[(3, 3)]} (High)")
print(f"  Consequence=2, Probability=2 → Risk = {risk_matrix_dict[(2, 2)]} (Medium)")
print(f"  Consequence=1, Probability=1 → Risk = {risk_matrix_dict[(1, 1)]} (Low)")


## Step 9: Calculate Risk Scores (Separately for Each Asset Group)

We apply the risk matrix **separately** for Asset 1 and Asset 2, using each group's
own consequence and probability scores.

This produces: `risk_a1` and `risk_a2`.

In [ ]:
# Apply the risk matrix separately for each asset group
parcels['risk_a1'] = parcels.apply(
    lambda row: risk_matrix_dict.get((row['consequence_a1'], row['probability_a1']), 0),
    axis=1
)
parcels['risk_a2'] = parcels.apply(
    lambda row: risk_matrix_dict.get((row['consequence_a2'], row['probability_a2']), 0),
    axis=1
)

print("Risk scores calculated for both asset groups!")
risk_labels = {3: 'High', 2: 'Medium', 1: 'Low', 0: 'No Data'}
for col, label in [('risk_a1', 'Asset 1'), ('risk_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label} ({len(in_group):,} parcels):")
    for score in sorted(in_group[col].unique(), reverse=True):
        count = (in_group[col] == score).sum()
        pct = 100 * count / len(in_group)
        print(f"  {risk_labels.get(score, '?')} ({score}): {count:,} ({pct:.1f}%)")

## Step 9.5: Risk Results Matrix with Cross-Tabulation Statistics

The **Risk Scoring Matrix** we saw earlier (Step 4) is a **reference tool** that tells us the lookup rules. Now we will create a **Results Matrix** that shows what actually happened in our data.

The Results Matrix visualizes a **cross-tabulation** of all parcels, grouped by their probability and consequence scores. For each cell:

- **Number of parcels** in that combination
- **Total parcel value** (land value) in millions of dollars
- **Total building value** (improvement value) in millions of dollars
- **Risk score** for that combination (color-coded: light lavender=Low, light purple=Medium, medium purple=High)

This results matrix helps answer questions like:
- "How many parcels are in the high-consequence, high-probability zone?"
- "What is the total property value at risk in the worst-case scenario?"
- "Which combinations of probability and consequence actually occur in our data?"

The rows are ordered **High to Low** (same as the reference matrix in Step 5) so you can easily compare the two tables.

In [ ]:
# Helper function to format currency values
def format_currency(val):
    """Format a numeric value as a currency string"""
    if pd.isna(val) or val == 0:
        return '$0'
    if abs(val) >= 1_000_000:
        return f'${val/1_000_000:.1f}M'
    if abs(val) >= 1_000:
        return f'${val/1_000:.1f}K'
    return f'${val:.0f}'

# Risk matrix lookup
risk_matrix_lookup = {
    (3, 1): 2, (3, 2): 3, (3, 3): 3,
    (2, 1): 1, (2, 2): 2, (2, 3): 3,
    (1, 1): 1, (1, 2): 1, (1, 3): 2,
}

consequence_levels = [3, 2, 1]  # High to Low (top to bottom)
probability_levels = [1, 2, 3]  # Low to High (left to right)

# Recalculate structure_value safely
parcels['structure_value'] = (parcels['parval'].fillna(0) - parcels['landval'].fillna(0)).clip(lower=0)

# Build results for each asset group
def build_results(df, cons_col, prob_col):
    results = {}
    for cons in consequence_levels:
        for prob in probability_levels:
            mask = (df[cons_col] == cons) & (df[prob_col] == prob)
            subset = df[mask]
            results[(cons, prob)] = {
                'count': len(subset),
                'parval': subset['parval'].fillna(0).sum(),
                'improvval': subset['structure_value'].fillna(0).sum(),
                'risk': risk_matrix_lookup.get((cons, prob), 0),
            }
    return results

a1_parcels = parcels[parcels['is_asset_1'] == 1]
a2_parcels = parcels[parcels['is_asset_2'] == 1]

results_a1 = build_results(a1_parcels, 'consequence_a1', 'probability_a1')
results_a2 = build_results(a2_parcels, 'consequence_a2', 'probability_a2')

# --- Draw side-by-side matrices ---
color_map = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}
text_color_map = {1: 'black', 2: 'black', 3: 'white'}

fig, axes = plt.subplots(1, 2, figsize=(22, 11))

for ax, results, title in zip(axes, [results_a1, results_a2],
                               ['Asset 1 Risk Results Matrix', 'Asset 2 Risk Results Matrix']):
    for i, cons in enumerate(consequence_levels):
        for j, prob in enumerate(probability_levels):
            data = results[(cons, prob)]
            risk_score = data['risk']
            color = color_map[risk_score]
            x, y = j, 2 - i
            rect = plt.Rectangle((x, y), 1, 1, linewidth=2,
                                 edgecolor='black', facecolor=color)
            ax.add_patch(rect)
            text_color = text_color_map[risk_score]
            cell_text = (
                f"Risk {risk_score}\n"
                f"{data['count']} parcels\n"
                f"{format_currency(data['parval'])} parcel val\n"
                f"{format_currency(data['improvval'])} bldg val"
            )
            ax.text(x + 0.5, y + 0.5, cell_text,
                    ha='center', va='center', fontsize=10, fontweight='bold',
                    color=text_color, wrap=True)
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect('equal')
    ax.set_xticks([0.5, 1.5, 2.5])
    ax.set_xticklabels(['Prob: Low (1)', 'Prob: Medium (2)', 'Prob: High (3)'],
                       fontsize=11, fontweight='bold')
    ax.set_yticks([0.5, 1.5, 2.5])
    ax.set_yticklabels(['Cons: Low (1)', 'Cons: Medium (2)', 'Cons: High (3)'],
                       fontsize=11, fontweight='bold')
    ax.set_xlabel('Probability \u2192', fontsize=12, fontweight='bold')
    ax.set_ylabel('Consequence \u2193', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\u2713 Risk Results Matrices created (Asset 1 and Asset 2)")
print(f"  Asset 1 parcels: {len(a1_parcels):,}")
print(f"  Asset 2 parcels: {len(a2_parcels):,}")


## Step 10: Analyze Risk Statistics

Let's count how many parcels fall into each risk category and calculate percentages. This summary helps us understand the overall risk level of the study area.


In [ ]:
# Risk summary statistics per asset group
risk_labels_map = {3: 'High (3)', 2: 'Medium (2)', 1: 'Low (1)', 0: 'No Data (0)'}

for col, flag, label in [('risk_a1', 'is_asset_1', 'Asset 1'),
                          ('risk_a2', 'is_asset_2', 'Asset 2')]:
    subset = parcels[parcels[flag] == 1]
    risk_counts = subset[col].value_counts().sort_index(ascending=False)

    print('=' * 60)
    print(f'RISK SUMMARY STATISTICS  --  {label}')
    print('=' * 60)
    for risk_val, count in risk_counts.items():
        pct = (count / len(subset)) * 100
        rlabel = risk_labels_map.get(risk_val, f'Unknown ({risk_val})')
        print(f"{rlabel:20} | Count: {count:5} | Percentage: {pct:6.2f}%")
    print('-' * 60)
    print(f"{'TOTAL':20} | Count: {len(subset):5} | Percentage: 100.00%")
    print('=' * 60)
    print()

print("\u2713 Risk summary complete (per asset group)")


In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

risk_colors = {1: '#D8D0E0', 2: '#B5A8C8', 3: '#7B6B95'}

# Create side-by-side figure: Asset 1 | Asset 2, shared legend below
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[10, 1.2], hspace=0.05, wspace=0.05)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax_legend = fig.add_subplot(gs[1, :])

for ax, risk_col, flag_col, title in [
    (ax1, 'risk_a1', 'is_asset_1', 'Risk Assessment -- Asset 1'),
    (ax2, 'risk_a2', 'is_asset_2', 'Risk Assessment -- Asset 2'),
]:
    # Layer 1: all parcels, no fill
    parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

    # Layer 2: risk scores for this asset group only
    group = parcels_wm[parcels_wm[flag_col] == 1]
    for score in [3, 2, 1]:
        subset = group[group[risk_col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, facecolor=risk_colors[score], edgecolor='none', alpha=0.85)

    # Layer 3: Buildings
    try:
        buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
        buildings_wm = buildings_layer.to_crs(epsg=3857)
        buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a',
                          linewidth=0.1, alpha=0.7)
    except Exception:
        pass

    # Layer 4: Flood zones
    try:
        flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
        flood_wm = flood_zones_full.to_crs(epsg=3857)
        flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
        for flood_type in ['500-year', '100-year', 'Floodway']:
            flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
            if len(flood_subset) > 0:
                flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                                 edgecolor='none', alpha=0.5)
    except Exception:
        pass

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')
    ax.set_axis_off()
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)

# Shared legend
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='#7B6B95', edgecolor='none', label='High (3)'),
    Patch(facecolor='#B5A8C8', edgecolor='none', label='Medium (2)'),
    Patch(facecolor='#D8D0E0', edgecolor='none', label='Low (1)'),
    Patch(facecolor='none', edgecolor='none', label=''),
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.5, label='Floodway'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.5, label='100-year Floodplain'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.5, label='500-year Floodplain'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                 frameon=True, facecolor='white', edgecolor='#cccccc',
                 handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
png_path = os.path.join(OUTPUT_DIR, 'risk_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"\u2713 Side-by-side risk maps exported to: {png_path}")


## Step 12.5: Export Risk Map as PNG

Now we'll create a publication-quality PNG map that combines all layers in the proper order with dark styling. This map shows risk assessment with all supporting layers (buildings, flood zones, and parcels) in a single exportable image.

## Step 12: Create a Chart of Risk Distribution

Let's visualize the risk distribution as a bar chart. This makes it easy to see at a glance how many parcels are in each risk category.


In [ ]:
# Bar chart of risk distribution per asset group
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

risk_order = [3, 2, 1, 0]
risk_labels_chart = ['High (3)', 'Medium (2)', 'Low (1)', 'No Data (0)']
risk_colors_chart = ['#7B6B95', '#B5A8C8', '#D8D0E0', '#CCCCCC']

for ax, risk_col, flag_col, title in [
    (axes[0], 'risk_a1', 'is_asset_1', 'Asset 1 Risk Distribution'),
    (axes[1], 'risk_a2', 'is_asset_2', 'Asset 2 Risk Distribution'),
]:
    subset = parcels[parcels[flag_col] == 1]
    counts = [len(subset[subset[risk_col] == r]) for r in risk_order]

    bars = ax.bar(risk_labels_chart, counts, color=risk_colors_chart,
                  edgecolor='black', linewidth=1.5)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax.set_xlabel('Risk Level', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Parcels', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\u2713 Risk distribution charts created (per asset group)")


## Save Results to GeoPackage

We save the separate risk scores (`risk_a1`, `risk_a2`) to the GeoPackage.
Class 8 will combine these with vulnerability to calculate the final combined score.

In [ ]:
import fiona
import sqlite3
import os

# Delete old output to prevent append-duplicates
if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

try:
    # Save parcels layer (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG')
    print(f"✓ Saved parcels layer ({len(parcels)} features)")

    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        output_layers = fiona.listlayers(OUTPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels' or layer_name in output_layers:
                continue
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                output_layers.append(layer_name)
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}': {e}")

        # Copy non-spatial tables via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass

    # Ensure base layers from Class 0 are included
    if os.path.exists(CLASS0_GPKG):
        try:
            output_layers = fiona.listlayers(OUTPUT_GPKG)
            class0_layers = fiona.listlayers(CLASS0_GPKG)
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass

    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving: {e}")

## Summary

You've completed **Class 7: Risk**.

**What You Accomplished:**
- Applied the risk matrix (Consequence x Probability) **separately** for Asset 1 and Asset 2
- Each asset group has its own risk score: `risk_a1`, `risk_a2`

**Next:** Class 8 (Combined Score) will combine vulnerability and risk into the final assessment.

## Color Reference for GIS Symbology

Use these hex color values when styling your risk layer in QGIS or ArcGIS Pro:

| Score | Label | Hex Code | RGB |
|-------|-------|----------|-----|
| 1 | Low | `#D8D0E0` | 216, 208, 224 |
| 2 | Medium | `#B5A8C8` | 181, 168, 200 |
| 3 | High | `#7B6B95` | 123, 107, 149 |

**How to apply in QGIS:**
1. Right-click your layer → Properties → Symbology
2. Choose "Categorized" from the dropdown
3. Set Column to `risk_a1` / `risk_a2`
4. Click "Classify"
5. Double-click each symbol to change its color using the hex values above

**How to apply in ArcGIS Pro:**
1. Right-click your layer → Symbology
2. Choose "Unique Values"
3. Set Field 1 to `risk_a1` / `risk_a2`
4. Click "Add all values"
5. Double-click each symbol to change its color using the hex values above

**Pre-made symbology files** are also available in the `symbology/` folder:
- `risk_symbology.qml` — Load in QGIS via Style → Load Style
- `risk_symbology.lyrx` — Import in ArcGIS Pro via Symbology → Import


## Appendix A: How to Do This in QGIS

If you want to calculate risk scores in QGIS instead of Python, use a **Field Calculator** with this expression:

```
CASE
  WHEN "consequence_a1" = 3 AND "probability_a1" = 1 THEN 2
  WHEN "consequence_a1" = 3 AND "probability_a1" = 2 THEN 3
  WHEN "consequence_a1" = 3 AND "probability_a1" = 3 THEN 3
  WHEN "consequence_a1" = 2 AND "probability_a1" = 1 THEN 1
  WHEN "consequence_a1" = 2 AND "probability_a1" = 2 THEN 2
  WHEN "consequence_a1" = 2 AND "probability_a1" = 3 THEN 3
  WHEN "consequence_a1" = 1 AND "probability_a1" = 1 THEN 1
  WHEN "consequence_a1" = 1 AND "probability_a1" = 2 THEN 1
  WHEN "consequence_a1" = 1 AND "probability_a1" = 3 THEN 2
  ELSE 0
END
```

### Steps in QGIS:
1. Load your parcel layer in QGIS
2. Open the attribute table (right-click layer → Open Attribute Table)
3. Click **Field Calculator** (function icon in toolbar)
4. Create a new field called `risk_a1` / `risk_a2` (type: Integer)
5. Paste the CASE expression above
6. Click OK to calculate for all features
7. Save the layer

The result will be identical to what we calculated in Python.


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## Appendix B: How to Do This in ArcGIS Pro

In ArcGIS Pro, you can create a custom function in Python and use it in the **Field Calculator**.

### Step 1: Create a Python Function
```python
def calc_risk(consequence, probability):
    """Calculate risk from consequence and probability"""
    matrix = {
        (3, 1): 2, (3, 2): 3, (3, 3): 3,
        (2, 1): 1, (2, 2): 2, (2, 3): 3,
        (1, 1): 1, (1, 2): 1, (1, 3): 2,
    }
    return matrix.get((consequence, probability), 0)
```

### Step 2: Use in Field Calculator
In the Field Calculator expression box, enter:
```
calc_risk(!consequence_a1!, !probability_a1!)
```

### Step 3: Set Result Type
- Result Type: **Short Integer** or **Integer**
- Field Name: `risk_a1` / `risk_a2`

### Step 4: Execute
Click **OK** to calculate the risk field for all features.

The result will match our Python and QGIS calculations exactly.

### Note:
If you want to use this in ArcGIS ModelBuilder or other workflows, you can save the function in a `.py` file in your Python Toolbox and call it from there.


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## Appendix C: Troubleshooting

### Problem: "GeoPackage file not found"
**Solution**:
- Check that the file `vulnerability_risk_data.gpkg` exists in your Google Drive folder `MSER_510_VULNERABILTY`
- Make sure you authorized Google Colab to access your Drive (Step 2)
- If needed, upload the file to the correct folder

### Problem: "No such layer: parcels"
**Solution**:
- The GeoPackage might have a different layer name
- Run this code to see what layers are available:
  ```python
  import fiona
  layers = fiona.listlayers(gpkg_path)
  print(layers)
  ```
- Change `'parcels'` to the correct layer name

### Problem: "KeyError" when looking up risk
**Solution**:
- Some parcels might have missing probability or consequence values (NaN)
- The code handles this by returning 0, which we label as "No Data"
- Check the data quality: `print(parcels[['probability', 'consequence']].describe())`

### Problem: Map doesn't display correctly
**Solution**:
- The map is saved to `risk_map.html` in your Colab workspace
- If geometries are not valid, try: `parcels = parcels[parcels.geometry.is_valid]`
- Make sure geometries are in WGS84 (EPSG:4326) for Folium: `parcels = parcels.to_crs('EPSG:4326')`
